<a href="https://colab.research.google.com/github/marlabharghavsai/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [3]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [14]:
from google import genai
from pydantic import BaseModel, ValidationError
from typing import List, Optional

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    # ── Input guard ──────────────────────────────────────────────
    if not raw_text or len(raw_text.strip()) < 50:
        # Raise ValidationError the same way Pydantic would on missing fields
        Resume.model_validate({})   # empty dict → triggers "Field required" for name, email, etc.

    # ── Output guard helper ──────────────────────────────────────
    def _validate_output(parsed: Resume) -> Resume:
        if not parsed.name or not parsed.name.strip():
            raise ValueError("Model returned empty name — input may be invalid")
        if '[' in parsed.name:
            raise ValueError(f"Model returned placeholder data: {parsed.name}")
        return parsed

    # ── API call with retry ──────────────────────────────────────
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=(
                    'Extract a Resume JSON from this text. '
                    'Return ONLY JSON, no markdown.\n\n' + raw_text
                ),
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            parsed = Resume.model_validate_json(resp.text)
            return _validate_output(parsed)

        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = (
                f'Fix this JSON to match schema. Errors: {e}. '
                f'Original: {resp.text}'
            )
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            parsed = Resume.model_validate_json(resp.text)
            return _validate_output(parsed)

In [11]:
# Load sample résumés from the lab kit
with open('/content/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 3 sample résumés

Résumé 1: Ravi Kumar — 7 skills, 0.5 years exp

Résumé 2: Sneha Reddy — 14 skills, 0.33 years exp

Résumé 3: Arun Pillai — 23 skills, 0.5 years exp

=== Full first result ===
{
  "name": "Ravi Kumar",
  "email": "ravi.kumar@gmail.com",
  "phone": "+91-9876543210",
  "education": [
    {
      "degree": "B.Tech in Computer Science and Engineering",
      "institution": "Aditya University, Andhra Pradesh",
      "year": 2025
    }
  ],
  "skills": [
    "Python",
    "Java",
    "C",
    "HTML",
    "CSS",
    "MySQL",
    "Git"
  ],
  "projects": [
    "Library Management System (Python, MySQL)",
    "Student Result Portal (HTML, CSS, Java Servlets)"
  ],
  "experience_years": 0.5
}


In [15]:
# Empty string — should fail gracefully, not crash
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except ValidationError as e:
    print('Caught gracefully: ValidationError')
    print('Message:', str(e)[:200])
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Caught gracefully: ValidationError
Message: 5 validation errors for Resume
name
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
email
  Field required
